In [ ]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import nibabel as nib
from scipy import stats
from collections import defaultdict

In [ ]:
# parcels_of_interest = {
#     # From previous resting state paper: 133, 172, 9

#     'theory' : [133,172,192,284,339,395],
#     'data_driven' : [9,67,176,183,231,242,341,343,348,380,386,389,391]
# }

with open("/data/zachkaras/fmri_model_data/intermediate_results/all_results.pkl", 'rb'):
    records = pickle.load(f)

parcels_of_interest = [9,133,172]

variables = {
    'GPA': 'GPA',
    'Age': 'age',
    'Years of Experience': 'years_experience',
    'Code Editing' : 'code_editing_rate',
    'Prose Editing' : 'prose_editing_rate',
    'Code Total Keystrokes' : 'code_total_keystrokes',
    'Prose Total Keystrokes' : 'prose_total_keystrokes',
    'Code More Structured' : 'Code_more_structured', 
    'Correct' : 'correct',
    'Complete' : 'complete',
    'Code Correct' : 'code_correct',
    'Code Complete' : 'code_complete'
}

demo_data = pd.read_csv("/home/zachkaras/fmri_model/master-survey-data.csv")
demo_data['years_experience'] = demo_data['semesters_experience'] / 2
demo_data = demo_data.set_index('id')

best_models = ['code-deepseek_6b-ndelays_10-look_ahead_by_0',
               'prose-starcoder2_7b-ndelays_16-look_ahead_by_3'
               ]

In [ ]:
# read in atlases
# atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"
atlas_base_path = "/home/zachkaras/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx] # contains the schaefer parcel numbers
cortex_vx = np.where(atlas_only_brain != 0)[0]
parcel_nums = atlas_only_brain[cortex_vx]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)

In [ ]:


class Regression_Info(object):
    # [model_name, task, look_ahead, n_delays, layer, stat]
    def __init__(self, model_name=None, task=None, look_ahead=None, n_delays=None, layer=None, stat=None):
        self.model_name = model_name
        self.task       = task
        self.look_ahead = look_ahead
        self.n_delays   = n_delays
        self.layer      = layer
        self.stat       = stat
        
    def __str__(self):
        return f"{self.model_name}, {self.task}, {self.look_ahead}, {self.n_delays}, {self.layer}, {self.stat}"

# def nested_dict():
#         return defaultdict(nested_dict)

# def convert_back_to_dict(d):
#     if isinstance(d, defaultdict):
#         return {k: convert_back_to_dict(v) for k, v in d.items()}
#     return d

# def convert_to_nifti(values):
#     # working backwards to save correlation values as voxels in MNI space
#     empty_schaefer[cortex_vx] = values
#     empty_mni[brain_idx] = empty_schaefer
#     result_brain = np.reshape(empty_mni, og_shape)

#     # Saving results
#     nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
#     nib.save(nifti_result, "test_plotting.nii.gz")
#     return result_brain, nifti_result


def parse_regression_info(path):
    
    parts = path.split('-') 
    # example: ['codegemma_7b', 'code', 'look_ahead_by_1', 'ndelays_0', 'layer_28', 'correlations.pkl']
    info = Regression_Info()
    info.model_name = parts[0]
    info.task       = parts[1]
    info.look_ahead = parts[2]
    info.n_delays   = parts[3]
    info.layer      = parts[4]
    info.stat       = (parts[5])[:-4]
    
    return info


# def find_cutoff(vec, threshold = 10**4):
    
#     # I can find the threshold point based on sorting, then keep everything in the same place
#     copy = vec.copy()
#     copy.sort()
#     cutoff = copy[-threshold-1]
#     return cutoff


def iterate_through_participants(filepath, stat):
    participants = os.listdir(filepath)
    
    records = []
    # iterating through participants
    
    for p in participants:
        print(p)
        datapath = f"{filepath}/{p}"
        files = os.listdir(datapath)
        stat_files = [f for f in files if re.search(stat, f) and not re.search(r'only_regressor|\+', f)]
        
        # iterating through the stat files
        # these are vectors of correlation coefficients between predicted and recorded signal
        # Different parameters were adjusted, so the folder contains all the possibilities
        # [model_name, task, look_ahead, n_delays, layer, stat]
        for sf in stat_files:
            
            info = parse_regression_info(sf)
            
            stat_file = f"{datapath}/{sf}"
            with open(stat_file, 'rb') as f:
                try:
                    stat_vec = pickle.load(f) # stat vec is just voxels from the schaefer parcel, about 130k voxels
                except:
                    print(f"issue with {p}: {sf}")
            
            # for the stat vec, I need to 
            
            # # filter to top 10k, 10k is default parameter but can be changed with threshold argument
            # cutoff = find_cutoff(stat_vec)
            # top_voxel_idx = (np.where(stat_vec > cutoff))[0]
            # top_voxel_vals = stat_vec[top_voxel_idx]
            
            # Using z-scored correlation coefficients for downstream correlation tests
            # z = np.arctanh(stat_vec)
            # cutoff = find_cutoff(z)
            # top_vals = z[np.where(z > cutoff)[0]]
            # participant_means = float(np.mean(top_vals))
            
            # top_parcels = parcel_nums[top_voxel_idx]
            # top_parcels = np.array([int(parcel) for parcel in top_parcels])
            
            new_record = {
                'participant' : p,
                'task' : info.task,
                'model' : info.model_name,
                'ndelays' : info.n_delays,
                'look_ahead' : info.look_ahead,
                'layer' : info.layer,
            }
            records.append(new_record)
            
    records = pd.DataFrame(records)
    return records


def main():
    # iterate through directories, parse the file names, and accumulate stats
    # parsing file names [model_name, task, look_ahead, n_delays, layer, stat]

    filepath = "/data2/zachkaras/fmri_model_data/ridge_regression_pca_params"
    
    records = iterate_through_participants(filepath, 'correlations')

    outpath = "/data/zachkaras/fmri_model_data/intermediate_results"
    
    with open(f"{outpath}/parcels_of_interest.pkl", 'wb') as f:
        pickle.dump(records, f)

if __name__ == "__main__":
    main()
        

# participant_base_path = "/data/zachkaras/fmri_model_data/ridge_regression_pca_params"
# participants = os.listdir(participant_base_path)

In [ ]:
records = []

for m in best_models:
    parts = m.split('-')
    task, model, delays, look_ahead = parts[0], parts[1], parts[2], parts[3]
    pattern = f"{model}-{task}-{look_ahead}-{delays}"

    for p in participants:
        files = os.listdir(f"{participant_base_path}/{p}")
        best_files = [f for f in files if re.search(pattern, f) and 'correlations' in f]

        # Accumulate per-layer means for each parcel, then average across layers
        layer_vals = defaultdict(list)  # (approach, parcel) -> [layer_mean, ...]

        for bf in best_files:
            with open(f"{participant_base_path}/{p}/{bf}", 'rb') as f:
                corrs = pickle.load(f)
            z_corrs = np.arctanh(corrs)

            for approach, parcels in parcels_of_interest.items():
                for parcel in parcels:
                    layer_vals[(approach, parcel)].append(
                        np.mean(find_parcel_voxels(parcel, z_corrs))
                    )

        # One row per participant × model × parcel, averaged across layers
        for (approach, parcel), vals in layer_vals.items():
            records.append({
                'model': m,
                'participant': int(p),
                'approach': approach,
                'parcel': parcel,
                'mean_z_corr': np.mean(vals),
            })

df = pd.DataFrame(records)
df = df.merge(demo_data[list(variables.values())], left_on='participant', right_index=True)

# Then stats are just:

from scipy import stats

stat_records = []
for (model, approach, parcel), group in df.groupby(['model', 'approach', 'parcel']):
    for label, col in variables.items():
        r, p = stats.spearmanr(group['mean_z_corr'], group[col], nan_policy='omit')
        stat_records.append({
            'model': model, 'approach': approach, 'parcel': parcel,
            'variable': label, 'r': r, 'p': p
        })

stats_df = pd.DataFrame(stat_records)